### Interactive Filename operation - Add/Substract/Replace


In [1]:
import os


class NameModifier:

    def __init__(self):
        self.parent_folder = None
        self.item_type = None
        self.delimiter = None
        self.operation = None
        self.index = None
        self.text = None
        self.sample_item = None
        self.replace_mode = None
        self.old_text = None

    # =========================================================
    # INPUT
    # =========================================================

    def get_parent_folder(self):
        """Ask for the folder containing the files/folders to rename."""

        while True:

            folder = input(
                "\nEnter the folder path containing the items: "
            ).strip().strip('"')

            if not os.path.isdir(folder):

                print(
                    f"\nError: Folder does not exist:"
                    f"\n{folder}"
                )

                continue

            self.parent_folder = folder
            break

    def get_item_type(self):
        """Ask whether files or folders should be processed."""

        print("\nWhat do you want to rename?")
        print("1. Files")
        print("2. Folders")

        while True:

            choice = input(
                "\nEnter your choice (1/2): "
            ).strip()

            if choice == "1":
                self.item_type = "file"
                return

            if choice == "2":
                self.item_type = "folder"
                return

            print("Please enter 1 or 2.")

    def get_delimiter(self):
        """
        Ask for the delimiter.

        Enter 'no' for no delimiter.
        """

        while True:

            delimiter = input(
                "\nEnter the delimiter "
                "(e.g. #), or type 'no' for no delimiter: "
            ).strip()

            if delimiter.lower() == "no":

                self.delimiter = None
                return

            if delimiter:

                self.delimiter = delimiter
                return

            print(
                "Delimiter cannot be empty. "
                "Use 'no' if there is no delimiter."
            )

    def get_sample_item(self):
        """Find the first matching file/folder to use as an example."""

        items = []

        for item in os.listdir(self.parent_folder):

            item_path = os.path.join(
                self.parent_folder,
                item
            )

            if self.item_type == "file":

                if os.path.isfile(item_path):
                    items.append(item)

            elif self.item_type == "folder":

                if os.path.isdir(item_path):
                    items.append(item)

        if not items:

            item_type = (
                "files"
                if self.item_type == "file"
                else "folders"
            )

            raise FileNotFoundError(
                f"No {item_type} were found in:\n"
                f"{self.parent_folder}"
            )

        self.sample_item = items[0]

    # =========================================================
    # NAME HANDLING
    # =========================================================

    def split_name(self, item_name):
        """
        Split an item name according to the selected delimiter.

        For files, the extension is kept separate.

        If delimiter is None ('no'), the complete name becomes
        index 0.
        """

        if self.item_type == "file":

            name, extension = os.path.splitext(item_name)

        else:

            # Folders have no extension
            name = item_name
            extension = ""

        # No delimiter
        if self.delimiter is None:

            parts = [name]

        else:

            parts = name.split(self.delimiter)

        return parts, extension

    def display_index_example(self):
        """Display the sample item with its available indices."""

        parts, extension = self.split_name(
            self.sample_item
        )

        print("\n" + "=" * 70)
        print("INDEX EXAMPLE")
        print("=" * 70)

        item_type = (
            "File"
            if self.item_type == "file"
            else "Folder"
        )

        print(f"\n{item_type}:")
        print(f"  {self.sample_item}")

        if self.delimiter is None:

            print("\nDelimiter: NONE")

        else:

            print(
                f"\nDelimiter: '{self.delimiter}'"
            )

        print("\nAvailable indices:")

        for index, part in enumerate(parts):

            print(
                f"  [{index}] {part}"
            )

        print("=" * 70)

        return parts

    # =========================================================
    # OPERATION
    # =========================================================

    def get_operation(self):
        """Ask which operation to perform."""

        print("\nChoose an operation:")
        print("1. Add")
        print("2. Subtract")
        print("3. Replace")

        while True:

            operation = input(
                "\nEnter your choice (1/2/3): "
            ).strip()

            if operation in ("1", "2", "3"):
                return operation

            print("Please enter 1, 2, or 3.")

    def get_index(self):
        """Display example and ask for an index."""

        parts, extension = self.split_name(
            self.sample_item
        )

        print("\nExample based on the item:")

        print(
            f"  {self.sample_item}"
        )

        print("\nIndices:")

        for index, part in enumerate(parts):

            if self.operation == "1":

                description = (
                    "text will be added to this part"
                )

            elif self.operation == "2":

                description = (
                    "this part will be removed"
                )

            else:

                description = (
                    "this part will be replaced"
                )

            print(
                f"  [{index}] {part} "
                f"<-- {description}"
            )

        print()

        while True:

            try:

                index = int(
                    input("Enter the index: ")
                )

                if index < 0:

                    print(
                        "Index must be 0 or greater."
                    )

                    continue

                if index >= len(parts):

                    print(
                        f"Index must be between "
                        f"0 and {len(parts) - 1}."
                    )

                    continue

                return index

            except ValueError:

                print(
                    "Please enter a valid integer."
                )

    def get_operation_text(self):
        """Get text required by Add or Replace."""

        if self.operation == "1":

            self.text = input(
                "\nEnter the text to ADD "
                "(e.g. _A): "
            )

        elif self.operation == "3":

            # Ask which replace mode
            print("\nReplace options:")
            print("1. Replace entire part")
            print("2. Replace specific text within the part")

            while True:

                mode = input(
                    "\nEnter your choice (1/2): "
                ).strip()

                if mode == "1":

                    self.replace_mode = "entire"
                    break

                if mode == "2":

                    self.replace_mode = "specific"
                    break

                print("Please enter 1 or 2.")

            if self.replace_mode == "entire":

                self.text = input(
                    "\nEnter the REPLACEMENT text "
                    "(e.g. 12456): "
                )

            else:
                # Specific text replacement
                self.old_text = input(
                    "\nEnter the text to FIND "
                    "(e.g. 01_08_1936): "
                )

                self.text = input(
                    "\nEnter the REPLACEMENT text "
                    "(e.g. 01_08_1938): "
                )

    # =========================================================
    # USER INPUT
    # =========================================================

    def get_user_input(self):
        """Collect all required user input."""

        # -----------------------------------------------------
        # 1. FILE OR FOLDER
        # -----------------------------------------------------

        self.get_item_type()

        # -----------------------------------------------------
        # 2. PARENT FOLDER
        # -----------------------------------------------------

        self.get_parent_folder()

        # -----------------------------------------------------
        # 3. DELIMITER
        # -----------------------------------------------------

        self.get_delimiter()

        # -----------------------------------------------------
        # Find sample item
        # -----------------------------------------------------

        self.get_sample_item()

        # -----------------------------------------------------
        # Show example
        # -----------------------------------------------------

        self.display_index_example()

        # -----------------------------------------------------
        # 4. OPERATION
        # -----------------------------------------------------

        self.operation = self.get_operation()

        # -----------------------------------------------------
        # 5. INDEX
        # -----------------------------------------------------

        self.index = self.get_index()

        # -----------------------------------------------------
        # 6. TEXT
        # -----------------------------------------------------

        self.get_operation_text()

    # =========================================================
    # MODIFY NAME
    # =========================================================

    def modify_name(self, item_name):
        """
        Modify a file or folder name.

        ADD:
            Adds text to the selected part.

        SUBTRACT:
            Removes the selected part.

        REPLACE:
            Replaces the selected part (entire) or
            replaces specific text within the selected part.
        """

        parts, extension = self.split_name(
            item_name
        )

        # -----------------------------------------------------
        # Validate index
        # -----------------------------------------------------

        if self.index >= len(parts):

            print(
                f"Skipping: {item_name}"
                f"\nIndex {self.index} does not exist."
            )

            return None

        # -----------------------------------------------------
        # ADD
        # -----------------------------------------------------

        if self.operation == "1":

            parts[self.index] = (
                parts[self.index] + self.text
            )

        # -----------------------------------------------------
        # SUBTRACT
        # -----------------------------------------------------

        elif self.operation == "2":

            parts.pop(self.index)

        # -----------------------------------------------------
        # REPLACE
        # -----------------------------------------------------

        elif self.operation == "3":

            if self.replace_mode == "entire":

                parts[self.index] = self.text

            else:
                # Replace specific text within the part
                if self.old_text in parts[self.index]:

                    parts[self.index] = parts[self.index].replace(
                        self.old_text,
                        self.text
                    )

                else:

                    print(
                        f"Skipping: {item_name}"
                        f"\nText '{self.old_text}' not found "
                        f"in part [{self.index}]."
                    )

                    return None

        # -----------------------------------------------------
        # Rebuild name
        # -----------------------------------------------------

        if self.delimiter is None:

            new_name = "".join(parts)

        else:

            new_name = self.delimiter.join(parts)

        # -----------------------------------------------------
        # Prevent empty names
        # -----------------------------------------------------

        if not new_name:

            print(
                f"Skipping: {item_name}"
                f"\nResulting name would be empty."
            )

            return None

        # -----------------------------------------------------
        # Add extension back for files
        # -----------------------------------------------------

        return new_name + extension

    # =========================================================
    # PROCESS ITEMS
    # =========================================================

    def process_items(self):
        """Rename all matching files/folders."""

        print("\n")
        print("=" * 70)
        print("PROCESSING")
        print("=" * 70)

        for item_name in os.listdir(
            self.parent_folder
        ):

            old_path = os.path.join(
                self.parent_folder,
                item_name
            )

            # -------------------------------------------------
            # Check item type
            # -------------------------------------------------

            if self.item_type == "file":

                if not os.path.isfile(old_path):
                    continue

            elif self.item_type == "folder":

                if not os.path.isdir(old_path):
                    continue

            # -------------------------------------------------
            # Create new name
            # -------------------------------------------------

            new_name = self.modify_name(
                item_name
            )

            if new_name is None:
                continue

            # -------------------------------------------------
            # Check if name changed
            # -------------------------------------------------

            if new_name == item_name:

                print(
                    f"\nSkipping: {item_name}"
                    f"\nReason: no change."
                )

                continue

            # -------------------------------------------------
            # New path
            # -------------------------------------------------

            new_path = os.path.join(
                self.parent_folder,
                new_name
            )

            # -------------------------------------------------
            # Check duplicate
            # -------------------------------------------------

            if os.path.exists(new_path):

                print(
                    f"\nSkipping:"
                    f"\n  {item_name}"
                    f"\n  -> {new_name}"
                    f"\nReason: target already exists."
                )

                continue

            # -------------------------------------------------
            # Rename
            # -------------------------------------------------

            try:

                os.rename(
                    old_path,
                    new_path
                )

                print(
                    f"\nRenamed:"
                    f"\n  {item_name}"
                    f"\n  -> {new_name}"
                )

            except Exception as e:

                print(
                    f"\nError processing "
                    f"{item_name}: {e}"
                )

        print("\n" + "=" * 70)
        print("Finished processing.")
        print("=" * 70)

    # =========================================================
    # RUN
    # =========================================================

    def run(self):
        """Run the complete program."""

        try:

            self.get_user_input()

            self.process_items()

        except FileNotFoundError as e:

            print(
                f"\nError: {e}"
            )

        except Exception as e:

            print(
                f"\nUnexpected error: {e}"
            )


# =============================================================
# MAIN
# =============================================================

if __name__ == "__main__":

    modifier = NameModifier()

    modifier.run()


What do you want to rename?
1. Files
2. Folders

INDEX EXAMPLE

File:
  0001#_#422_BS_4267_11#ID0001#08_09_1945.tif

Delimiter: '#'

Available indices:
  [0] 0001
  [1] _
  [2] 422_BS_4267_11
  [3] ID0001
  [4] 08_09_1945

Choose an operation:
1. Add
2. Subtract
3. Replace

Example based on the item:
  0001#_#422_BS_4267_11#ID0001#08_09_1945.tif

Indices:
  [0] 0001 <-- this part will be replaced
  [1] _ <-- this part will be replaced
  [2] 422_BS_4267_11 <-- this part will be replaced
  [3] ID0001 <-- this part will be replaced
  [4] 08_09_1945 <-- this part will be replaced


Replace options:
1. Replace entire part
2. Replace specific text within the part


PROCESSING

Renamed:
  0001#_#422_BS_4267_11#ID0001#08_09_1945.tif
  -> 0001#_#422_BS_4266_21#ID0001#08_09_1945.tif

Renamed:
  0002#_#422_BS_4267_11#ID0001#08_09_1945.tif
  -> 0002#_#422_BS_4266_21#ID0001#08_09_1945.tif

Renamed:
  0003#_#422_BS_4267_11#ID0001#08_09_1945.tif
  -> 0003#_#422_BS_4266_21#ID0001#08_09_1945.tif

Rena

### Filename Rename - from a field in Excel sheet - 13.05.2025

In [ ]:
import os
import pandas as pd
from pathlib import Path

# Configuration
excel_path = 'V:\Archive\Landesarchive\Hauptstaatsarchiv Stuttgart\Luftbildpläne Scanauftrag\_Luftbildpläne_Verzeichnis.xlsx'  # Update with your Excel file path
sheet_name = 'Sheet1'               # Update if your sheet has a different name
image_folder = r'V:\Archive\Landesarchive\Hauptstaatsarchiv Stuttgart\Luftbildpläne Scanauftrag'

def rename_images_based_on_excel(excel_path, sheet_name, image_folder):
    """
    Rename image files based on mapping from Excel (Index -> NF)
    
    Args:
        excel_path: Path to the Excel file
        sheet_name: Name of the sheet containing the mapping
        image_folder: Path to the folder containing images to rename
    """
    try:
        # Read the Excel file
        df = pd.read_excel(excel_path, sheet_name=sheet_name)
        
        # Verify required columns exist
        if 'Index' not in df.columns or 'NF' not in df.columns:
            raise ValueError("Excel sheet must contain both 'Index' and 'NF' columns")
        
        # Create mapping dictionary (Index -> NF)
        name_mapping = dict(zip(df['Index'], df['NF']))
        
        # Get all image files in the folder
        image_files = [f for f in os.listdir(image_folder) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'))]
        
        # Rename files
        rename_count = 0
        for old_name in image_files:
            # Extract the index from the filename (assuming filename contains the index)
            # This part might need adjustment based on your actual filename format
            try:
                # Example if filename is "12345.jpg" where 12345 is the index
                index = Path(old_name).stem  # gets filename without extension
                
                # If your filenames are more complex, you might need to parse differently
                # For example, if they're "IMG_12345.jpg", you might need:
                # index = old_name.split('_')[1].split('.')[0]
                
                if index in name_mapping:
                    new_name = f"{name_mapping[index]}{Path(old_name).suffix}"
                    old_path = os.path.join(image_folder, old_name)
                    new_path = os.path.join(image_folder, new_name)
                    
                    # Check if new filename already exists
                    if os.path.exists(new_path):
                        print(f"Warning: {new_path} already exists. Skipping rename of {old_name}")
                        continue
                    
                    os.rename(old_path, new_path)
                    print(f"Renamed: {old_name} -> {new_name}")
                    rename_count += 1
            except Exception as e:
                print(f"Error processing {old_name}: {str(e)}")
        
        print(f"\nRenaming complete. {rename_count} files were renamed.")
        return rename_count
    
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        return 0

# Execute the renaming
rename_images_based_on_excel(excel_path, sheet_name, image_folder)

### Filename Rename - from a field in Excel sheet - 14.05.2025 (used for TK25_Luftbildkarte)

In [ ]:
import os

# Specify the directory where the files are located
folder_path = r'W:\_Umbenennung\Neuer Ordner'

# Loop through all the files in the specified folder
for filename in os.listdir(folder_path):
    # Split the filename by #
    parts = filename.split('#')
    
    # Find and replace the TK25 part
    for i, part in enumerate(parts):
        if part.startswith('TK25_'):
            parts[i] = 'TK25_Luftbildkarte'
    
    # Create new filename by joining with #
    new_filename = '#'.join(parts)
    
    # Only proceed if the filename actually changed
    if new_filename != filename:
        old_file = os.path.join(folder_path, filename)
        new_file = os.path.join(folder_path, new_filename)
        
        # Check if the new filename already exists
        if os.path.exists(new_file):
            print(f"Skipping: {new_file} already exists.")
            continue
        
        # Rename the file
        os.rename(old_file, new_file)
        print(f'Renamed: {filename} -> {new_filename}')

print("Finished renaming files.")

### Filename Rename - Rearrange date order in the filename (eg 1943_08_11 to 11_08_1943) - 25.06.2025

In [ ]:
import os
import re

# Set the target directory
directory = r'W:\1943_08_11#00511#D_0966\Neuer Ordner'

# Loop through all files in the directory
for filename in os.listdir(directory):
    # Search for the date pattern YYYY_MM_DD
    match = re.search(r'(\d{4})_(\d{2})_(\d{2})', filename)
    if match:
        yyyy, mm, dd = match.groups()
        old_date = f'{yyyy}_{mm}_{dd}'
        new_date = f'{dd}_{mm}_{yyyy}'
        
        # Create new filename
        new_filename = filename.replace(old_date, new_date)
        
        # Full old and new file paths
        old_path = os.path.join(directory, filename)
        new_path = os.path.join(directory, new_filename)
        
        # Rename the file
        os.rename(old_path, new_path)
        print(f'Renamed: {filename} -> {new_filename}')


### Adding Übergabe Lektorat Date(csv input) in the LBA_Bilder_In_Bestand Filenames

In [2]:
import pandas as pd
import os
import re
from datetime import datetime

class FolderRenamer:
    def __init__(self, file_path, target_dir):
        self.file_path = file_path
        self.target_dir = target_dir
        self.matched = []
        self.unmatched = []
        self.df = None
    
    def load_data(self):
        """Load CSV or XLSX file with automatic encoding detection"""
        file_ext = os.path.splitext(self.file_path)[1].lower()
        
        if file_ext == '.csv':
            return self._load_csv()
        elif file_ext in ['.xlsx', '.xls']:
            return self._load_excel()
        else:
            raise Exception(f"Unsupported file format: {file_ext}. Please use .csv, .xlsx, or .xls")
    
    def _load_csv(self):
        """Load CSV with automatic encoding and delimiter detection"""
        encodings = ['utf-8-sig', 'latin-1', 'cp1252', 'iso-8859-1', 'utf-16']
        delimiters = [',', '\t', ';', '|']
        
        for enc in encodings:
            for delim in delimiters:
                try:
                    self.df = pd.read_csv(self.file_path, encoding=enc, sep=delim)
                    if self._validate_and_clean():
                        print(f"✓ Loaded CSV using encoding: {enc}, delimiter: {repr(delim)}")
                        return True
                except:
                    continue
        
        raise Exception("Could not read CSV with any encoding/delimiter combination")
    
    def _load_excel(self):
        """Load Excel file"""
        try:
            # Try reading with openpyxl for .xlsx or xlrd for .xls
            self.df = pd.read_excel(self.file_path, engine='openpyxl' if self.file_path.endswith('.xlsx') else 'xlrd')
            if self._validate_and_clean():
                print(f"✓ Loaded Excel file: {os.path.basename(self.file_path)}")
                return True
        except Exception as e:
            # Fallback: try reading all sheets and use first one with data
            try:
                xl = pd.ExcelFile(self.file_path)
                for sheet_name in xl.sheet_names:
                    self.df = pd.read_excel(self.file_path, sheet_name=sheet_name)
                    if self._validate_and_clean():
                        print(f"✓ Loaded Excel sheet: {sheet_name}")
                        return True
            except:
                pass
            raise Exception(f"Could not read Excel file: {str(e)}")
    
    def _validate_and_clean(self):
        """Validate and clean the dataframe"""
        # Try to find the right columns
        rename_map = {}
        for col in self.df.columns:
            col_str = str(col).strip()
            if 'Projekt' in col_str and ('Nr' in col_str or 'Nr.' in col_str):
                rename_map[col] = 'Projekt Nr.'
            elif ('Übergabe' in col_str or 'Ubergabe' in col_str) and 'Lektorat' in col_str:
                rename_map[col] = 'Übergabe Lektorat'
        
        if rename_map:
            self.df = self.df.rename(columns=rename_map)
        
        # Check if we have the required columns
        if 'Projekt Nr.' in self.df.columns and 'Übergabe Lektorat' in self.df.columns:
            # Clean the data
            self.df = self.df.dropna(subset=['Projekt Nr.'])
            self.df['Übergabe Lektorat'] = pd.to_datetime(self.df['Übergabe Lektorat'], errors='coerce')
            self.df = self.df.dropna(subset=['Übergabe Lektorat'])
            return True
        
        return False
    
    def process(self):
        """Main processing: match folders with projects"""
        self.load_data()
        
        print("\nFirst 5 rows of data:")
        print(self.df[['Projekt Nr.', 'Übergabe Lektorat']].head())
        print()
        
        # Build lookup
        lookup = {}
        for _, row in self.df.iterrows():
            proj = str(row['Projekt Nr.']).strip()
            date_val = row['Übergabe Lektorat']
            if pd.notna(date_val):
                lookup[proj] = date_val
        
        print(f"✓ Loaded {len(lookup)} valid project-date mappings")
        
        # Check directory
        if not os.path.exists(self.target_dir):
            raise FileNotFoundError(f"Directory not found: {self.target_dir}")
        
        # Get all items
        all_items = os.listdir(self.target_dir)
        print(f"\n📁 Scanning: {self.target_dir}")
        print(f"Total items: {len(all_items)}")
        
        # Separate files and folders
        folders = []
        files = []
        for item in all_items:
            item_path = os.path.join(self.target_dir, item)
            if os.path.isdir(item_path):
                folders.append(item)
            else:
                files.append(item)
        
        print(f"  📁 Folders: {len(folders)}")
        print(f"  📄 Files: {len(files)} (will be ignored)")
        
        # Show first 10 folders as sample
        if folders:
            print("\nFirst 10 folders in directory:")
            for i, folder in enumerate(folders[:10], 1):
                print(f"  {i:2}. 📁 {folder}")
            if len(folders) > 10:
                print(f"  ... and {len(folders)-10} more folders")
        
        if not folders:
            print("\n⚠️ No folders found in directory!")
            return [], []
        
        print(f"\nProcessing {len(folders)} folders...")
        
        for folder_name in folders:
            # Extract project number from folder name
            match = re.search(r'([A-Z]{2}\d{4}-\d{4})', folder_name)
            proj_num = match.group(1) if match else None
            
            if proj_num and proj_num in lookup:
                date_str = lookup[proj_num].strftime('%Y-%m-%d')
                # Check if date is already at the beginning
                if folder_name.startswith(date_str):
                    self.matched.append((folder_name, folder_name, proj_num, 'ALREADY CORRECT'))
                else:
                    new_name = f"{date_str} {folder_name}"
                    self.matched.append((folder_name, new_name, proj_num, 'TO RENAME'))
            else:
                self.unmatched.append((folder_name, proj_num))
        
        # Show results
        print(f"\n{'='*80}")
        to_rename = [m for m in self.matched if m[3] == 'TO RENAME']
        already_correct = [m for m in self.matched if m[3] == 'ALREADY CORRECT']
        print(f"SUMMARY:")
        print(f"  ✅ Folders to rename: {len(to_rename)}")
        print(f"  ✓ Already correct: {len(already_correct)}")
        print(f"  ❌ Unmatched: {len(self.unmatched)}")
        print(f"{'='*80}")
        
        if to_rename:
            print("\nFOLDERS TO RENAME:")
            print("-"*80)
            for i, (old, new, proj, status) in enumerate(to_rename, 1):
                print(f"{i:3}. {old}")
                print(f"     → {new} (Project: {proj})")
                print()
        
        if already_correct:
            print("\nFOLDERS ALREADY CORRECT (will not be changed):")
            print("-"*80)
            for old, new, proj, status in already_correct[:5]:
                print(f"  ✓ {old}")
            if len(already_correct) > 5:
                print(f"  ... and {len(already_correct)-5} more")
        
        if self.unmatched:
            print("\nUNMATCHED FOLDERS (will NOT be renamed):")
            print("-"*80)
            for folder, proj in self.unmatched[:10]:
                if proj:
                    print(f"  • {folder} (Project: {proj} - not in data)")
                else:
                    print(f"  • {folder} (No project number found)")
            if len(self.unmatched) > 10:
                print(f"  ... and {len(self.unmatched)-10} more")
        
        # Ask for confirmation
        if to_rename:
            print(f"\n{'='*80}")
            confirm = input(f"Rename {len(to_rename)} folders? (yes/no): ").strip().lower()
            if confirm in ['yes', 'y']:
                self._rename_folders(to_rename)
            else:
                print("❌ Cancelled")
        else:
            print("\n✅ No folders need renaming!")
        
        return self.matched, self.unmatched
    
    def _rename_folders(self, to_rename):
        """Execute folder renaming"""
        # Create backup
        backup_path = os.path.join(self.target_dir, f"backup_folders_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
        with open(backup_path, 'w', encoding='utf-8') as f:
            f.write("BACKUP OF RENAMED FOLDERS\n")
            f.write("="*80 + "\n\n")
            for old, new, proj, status in to_rename:
                f.write(f"Project: {proj}\n")
                f.write(f"Old: {old}\n")
                f.write(f"New: {new}\n")
                f.write("-"*40 + "\n")
        print(f"✓ Backup saved: {backup_path}")
        
        # Rename folders
        renamed, errors = 0, 0
        print("\nRenaming folders:")
        print("-"*80)
        
        for old, new, proj, status in to_rename:
            old_path = os.path.join(self.target_dir, old)
            new_path = os.path.join(self.target_dir, new)
            try:
                if not os.path.exists(new_path):
                    os.rename(old_path, new_path)
                    renamed += 1
                    print(f"  ✓ {old} → {new}")
                else:
                    print(f"  ✗ SKIP: {new} already exists")
                    errors += 1
            except Exception as e:
                print(f"  ✗ ERROR: {old} - {e}")
                errors += 1
        
        print(f"\n{'='*80}")
        print(f"COMPLETE: {renamed} renamed, {errors} errors")
        print(f"{'='*80}")

# Usage
if __name__ == "__main__":
    # Can be .csv, .xlsx, or .xls
    DATA_FILE = r"G:\0 BIPLOV\TASKS\Patrick\ShortcutPrepFromHarmonyProjects\PROCESSING\projektliste.xlsx"  # Change to your file
    DIR = r"G:\0 BIPLOV\TASKS\Patrick\ShortcutPrepFromHarmonyProjects\Neuer Ordner"
    
    renamer = FolderRenamer(DATA_FILE, DIR)
    renamer.process()

✓ Loaded Excel file: projektliste.xlsx

First 5 rows of data:
     Projekt Nr. Übergabe Lektorat
0    BE2026-1024        2026-08-06
1    BE2026-1047        2026-08-07
2    BE2026-1218        2026-08-07
3    BE2026-1211        2026-08-10
4  BE2026-1001_2        2026-08-11

✓ Loaded 51 valid project-date mappings

📁 Scanning: G:\0 BIPLOV\TASKS\Patrick\ShortcutPrepFromHarmonyProjects\Neuer Ordner
Total items: 7
  📁 Folders: 7
  📄 Files: 0 (will be ignored)

First 10 folders in directory:
   1. 📁 BE2026-0991 Freiamt L 110 Straßenarbeiten_shortcuts
   2. 📁 BE2026-1052 Esslingen Eichendorffstraße 62_shortcuts
   3. 📁 BE2026-1187 Gerlingen Adolph-Kolping-Weg Leitungsarbeiten_shortcuts
   4. 📁 BE2026-1210 Sindelfingen Bachstraße 26_shortcuts
   5. 📁 BE2026-1225 Waiblingen Ortsmitte Straßenarbeiten_shortcuts
   6. 📁 BE2026-1269 Hambrücken Gemarkung Hambrücken_shortcuts
   7. 📁 BE2026-1270 Markgröningen Gemarkung Markgröningen_shortcuts

Processing 7 folders...

SUMMARY:
  ✅ Folders to rename: 7